# Ridge refraction — TsunamiTrace example

Tsunami ray refraction across a synthetic N–S submarine ridge.
A Gaussian ridge sits between the source (eastern deep ocean) and the western
edge of the domain.  The shallow ridge crest slows the wave
($c = \sqrt{g\,d}$), bending rays toward the normal to the isobaths
(Snell's law on a sphere).

| Location | Depth | Wave speed |
|---|---|---|
| Deep ocean | 5 000 m | ≈ 221 m s⁻¹ |
| Ridge crest | 400 m | ≈ 63 m s⁻¹ |

**Reference:** Gusman, A. R., Satake, K., Shinohara, M., Sakai, S. I., & Tanioka, Y. (2017).
Fault slip distribution of the 2016 Fukushima earthquake estimated from tsunami waveforms.
*Pure and Applied Geophysics*, 174(8), 2925–2943.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
%matplotlib inline
import TsunamiTrace as tt

## Grid

In [2]:
N_LON, N_LAT = 350, 400
lon_arr = np.linspace(130.0, 165.0, N_LON)   # degrees east
lat_arr = np.linspace(-20.0,  20.0, N_LAT)   # degrees north

# meshgrid in (n_lat, n_lon) row-major order for contourf / contour
LON, LAT = np.meshgrid(lon_arr, lat_arr)

## Bathymetry

Deep-ocean background with a Gaussian N–S ridge.
The ridge runs the full height of the domain ($\sigma_\mathrm{lat} \gg$ domain half-height)
so rays cannot slip around the ends — refraction across the ridge dominates.

In [3]:
DEEP_OCEAN = 5_000.0   # m — background depth
RIDGE_SILL =   400.0   # m — shallowest depth at the ridge crest
RIDGE_LON  =   147.0   # degrees east — ridge centre longitude
SIGMA_LON  =     1.5   # degrees — ridge half-width in longitude
SIGMA_LAT  =    30.0   # degrees — ridge half-length (longer than domain)

ridge_rise = (DEEP_OCEAN - RIDGE_SILL) * np.exp(
    -(LON - RIDGE_LON)**2 / (2 * SIGMA_LON**2)
    - LAT**2              / (2 * SIGMA_LAT**2)
)
Z     = DEEP_OCEAN - ridge_rise   # (n_lat, n_lon) — row-major for plotting
depth = Z.T                       # (n_lon, n_lat) — column-major for trace_rays

print(f"Ridge crest depth : {depth.min():.0f} m  "
      f"(wave speed {np.sqrt(9.8 * depth.min()):.1f} m/s)")
print(f"Deep ocean depth  : {DEEP_OCEAN:.0f} m  "
      f"(wave speed {np.sqrt(9.8 * DEEP_OCEAN):.1f} m/s)")

Ridge crest depth : 402 m  (wave speed 62.8 m/s)
Deep ocean depth  : 5000 m  (wave speed 221.4 m/s)


## Source and integration parameters

In [4]:
SOURCE_LON = 157.0   # degrees east — placed in the deep ocean east of the ridge
SOURCE_LAT =   0.0   # degrees north

# At deep-ocean speed (~221 m/s) the wavefront travels ~1 590 km in 2 hours,
# enough to cross the ridge (source-to-ridge ≈ 1 110 km at the equator).
DT           =  30          # s — time step
MAX_TIME     = 14_400       # s — 4 hours
AZIMUTHS_DEG = np.arange(0, 360, 2, dtype=float)   # every 2°, 180 rays

print(f"Source : {SOURCE_LON}°E, {SOURCE_LAT}°N")
print(f"Tracing {len(AZIMUTHS_DEG)} rays  (dt = {DT} s, max {MAX_TIME // 3600} h)")

Source : 157.0°E, 0.0°N
Tracing 180 rays  (dt = 30 s, max 4 h)


## Trace rays

In [5]:
ray_lon, ray_lat, _ = tt.trace_rays(
    lon_arr, lat_arr, depth,
    DT, MAX_TIME,
    SOURCE_LON, SOURCE_LAT,
    AZIMUTHS_DEG,
)

n_done = int(np.sum(~np.isnan(ray_lon[:, -1])))
print(f"{n_done}/{len(AZIMUTHS_DEG)} rays reached max_time; "
      f"{len(AZIMUTHS_DEG) - n_done} terminated early (boundary).")

AttributeError: module 'TsunamiTrace' has no attribute 'trace_rays'

## Visualise

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

# Filled depth contours
levels = np.linspace(RIDGE_SILL, DEEP_OCEAN, 30)
cf = ax.contourf(lon_arr, lat_arr, Z, levels=levels, cmap='Blues_r', alpha=0.85)

# Structural contours at round-number depths
depth_lines = [500, 1_000, 2_000, 3_000, 4_000]
cs = ax.contour(lon_arr, lat_arr, Z,
                levels=depth_lines, colors='steelblue',
                linewidths=0.7, alpha=0.6)
ax.clabel(cs, fmt='%d m', fontsize=7, inline=True)

# Ray paths — coloured by azimuth so individual rays are distinguishable
cmap_rays = plt.cm.plasma
for i in range(len(AZIMUTHS_DEG)):
    c = cmap_rays(i / len(AZIMUTHS_DEG))
    ax.plot(ray_lon[i], ray_lat[i], color=c, linewidth=0.5, alpha=0.7)

# Source marker
ax.plot(SOURCE_LON, SOURCE_LAT,
        marker=(5, 1), markersize=14,
        markerfacecolor='yellow', markeredgecolor='k', markeredgewidth=0.8,
        linestyle='none', zorder=6, label='source')

# Ridge crest line
ax.axvline(RIDGE_LON, color='w', linewidth=1.0, linestyle='--',
           alpha=0.6, label=f'ridge crest ({RIDGE_LON}°E)')

ax.set_xlim(lon_arr[0], lon_arr[-1])
ax.set_ylim(lat_arr[0], lat_arr[-1])
ax.set_xlabel('Longitude (°E)')
ax.set_ylabel('Latitude (°N)')
ax.set_title(
    f'Tsunami ray refraction — N–S submarine ridge\n'
    f'{len(AZIMUTHS_DEG)} rays, dt={DT} s, '
    f'max {MAX_TIME // 3600} h, '
    f'ridge crest {RIDGE_SILL:.0f} m / deep ocean {DEEP_OCEAN:.0f} m'
)
ax.legend(loc='lower left', fontsize=8)
ax.set_aspect('equal')

cbar = fig.colorbar(cf, ax=ax, fraction=0.025, pad=0.02)
cbar.set_label('Depth (m)')
cbar.ax.yaxis.set_major_formatter(
    ticker.FuncFormatter(lambda x, _: f'{x:,.0f}')
)

plt.tight_layout()
plt.show()